# 🌦️ Weather Trend Forecasting — Global Weather Repository

**PM Accelerator Technical Assessment**

> **PM Accelerator Mission:** The Product Manager Accelerator Program supports
> PM professionals through every stage of their careers — from students seeking
> entry-level roles to Directors stepping into leadership — through certified
> training, hands-on projects and career services, while fostering a diverse
> tech industry. Learn more: https://www.pmaccelerator.io/

This notebook walks through **data cleaning, EDA, forecasting (multiple models
+ ensemble) and advanced analyses** on the Global Weather Repository dataset.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import advanced_analysis, config, data_cleaning, eda, forecasting

pd.set_option("display.max_columns", 60)

## 1. Data Cleaning & Preprocessing
Parse `last_updated`, drop duplicates, impute missing values, winsorize
outliers (IQR), add normalized + calendar features.

In [ ]:
df = data_cleaning.clean_pipeline()
overview = data_cleaning.basic_overview(df)
overview

In [ ]:
# Missing-value report on the *raw* data (before imputation).
raw = data_cleaning.load_raw()
data_cleaning.missing_summary(raw).head(15)

## 2. Exploratory Data Analysis
Descriptive stats, temperature & precipitation visualizations, correlations,
and global trends.

In [ ]:
eda.numeric_summary(df)

In [ ]:
eda_figs = eda.run_all(df)
eda_figs

Figures are saved under `outputs/figures/`. Display a couple inline:

In [ ]:
from IPython.display import Image, display

for key in ("temperature_distribution", "precipitation_overview",
            "correlation_heatmap", "global_trend"):
    if key in eda_figs:
        display(Image(eda_figs[key]))

## 3. Forecasting — Multiple Models + Ensemble
Build a daily global-mean temperature series from `last_updated`, compare
Naive / Linear / Random Forest / SARIMAX, and an ensemble. Metrics: MAE,
RMSE, MAPE, R².

In [ ]:
series = forecasting.build_daily_series(df)
metrics_df, bundle = forecasting.evaluate_models(series)
metrics_df.round(3)

In [ ]:
forecasting.plot_forecast_comparison(bundle)
forecasting.plot_metrics(metrics_df)
display(Image(str(config.FIG_DIR / "forecast_comparison.png")))
display(Image(str(config.FIG_DIR / "model_metrics.png")))

### 30-day forward forecast (best statistical model refit on all data)

In [ ]:
future = forecasting.forecast_future(series)
forecasting.plot_future(series, future)
display(Image(str(config.FIG_DIR / "future_forecast.png")))
future.head(10)

## 4. Advanced Analyses
Anomaly detection, climate analysis, air-quality/environmental impact,
feature importance, and spatial/geographical patterns.

In [ ]:
adv_figs = advanced_analysis.run_all(df)
for path in adv_figs.values():
    display(Image(path))

## 5. Key Insights
See `report/REPORT.md` for the full written analysis. The pipeline also writes
`outputs/summary.json`, `outputs/model_metrics.csv` and
`outputs/future_forecast.csv` for reviewers.